# Collect All Results

Run this notebook after the three parts finish training.
It consolidates every saved `metrics.json` file into a single comparison table.


## Optional Google Colab Bootstrap

If you open this notebook directly from GitHub in Google Colab, run the next cell after:

1. Setting `GITHUB_REPO_URL`
2. Changing `RUN_COLAB_BOOTSTRAP` to `True`

Leave the cell as-is for local Jupyter use.


In [ ]:
import os
import subprocess
from pathlib import Path

RUN_COLAB_BOOTSTRAP = False
GITHUB_REPO_URL = ""
USE_GOOGLE_DRIVE = True
COLAB_REPO_DIR = "/content/drive/MyDrive/hi-192-dental-xray"

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and RUN_COLAB_BOOTSTRAP:
    if USE_GOOGLE_DRIVE:
        drive.mount("/content/drive")

    repo_root = Path(COLAB_REPO_DIR if USE_GOOGLE_DRIVE else "/content/hi-192-dental-xray")
    repo_root.parent.mkdir(parents=True, exist_ok=True)

    if not (repo_root / ".git").exists():
        if not GITHUB_REPO_URL.strip():
            raise ValueError("Set GITHUB_REPO_URL before running the Colab bootstrap cell.")
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(repo_root)], check=True)

    os.chdir(repo_root)
    print(f"Colab working directory set to: {repo_root}")
else:
    print(f"Current working directory: {Path.cwd()}")


In [ ]:
from pathlib import Path
import sys

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "dental_opg_experiment.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. In Colab, run the bootstrap cell first or clone the repo manually."
    )

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import dental_opg_experiment as doe


In [ ]:
from pathlib import Path

DATASET_ROOT = None
OUTPUTS_DIR = None

resolved_dataset_root = doe.resolve_dataset_root(DATASET_ROOT)
resolved_outputs_dir = doe.resolve_outputs_dir(OUTPUTS_DIR)

print(f"Dataset root: {resolved_dataset_root}")
print(f"Outputs dir: {resolved_outputs_dir}")


In [ ]:
all_results = doe.aggregate_part_results(outputs_dir=OUTPUTS_DIR)
all_results


In [ ]:
if not all_results.empty:
    output_path = doe.resolve_outputs_dir(OUTPUTS_DIR) / "all_results_summary.csv"
    all_results.to_csv(output_path, index=False)
    display(all_results.sort_values("accuracy", ascending=False).head(20))
    print(f"Saved combined results to: {output_path}")
else:
    print("No metrics were found yet.")
